In [ ]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from tqdm.auto import tqdm
import joblib
from contextlib import contextmanager


# =====================================================
# USER SETTINGS
# =====================================================

SEED = 42
np.random.seed(SEED)

SUBSET_SIZE = 1000
TEST_SIZE = 0.2
N_ITER_SEARCH = 25
CV_FOLDS = 5

TARGETS = ["band_gap", "s_n"]   # ← RUN BOTH SEQUENTIALLY

# =====================================================
# PROGRESS BAR PATCH
# =====================================================

@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

# =====================================================
# LOGGER
# =====================================================

def log_progress(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}")

# =====================================================
# LOAD DATA ONCE
# =====================================================

log_progress("Loading dataset...")
df_full = pd.read_csv("train_embeddings_250epochs_embdim128_original.csv")

feature_cols = [c for c in df_full.columns if c.startswith("feature")]

if SUBSET_SIZE is not None and SUBSET_SIZE < len(df_full):
    log_progress(f"Selecting random subset of {SUBSET_SIZE} structures...")
    df_full = df_full.sample(n=SUBSET_SIZE, random_state=SEED).reset_index(drop=True)

log_progress(f"Dataset ready: {len(df_full)} samples")

# =====================================================
# GENERIC MODEL RUNNER
# =====================================================

def run_target(TARGET):

    log_progress(f"\n==============================")
    log_progress(f"Starting training for TARGET: {TARGET}")
    log_progress(f"==============================\n")

    X = df_full[feature_cols].values
    y = df_full[TARGET].values

    # -------------------------
    # Train / Test split
    # -------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED
    )

    # -------------------------
    # Scaling
    # -------------------------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # -------------------------
    # Evaluation
    # -------------------------
    def evaluate(model, name):
        y_pred = model.predict(X_test)
        return {
            "Target": TARGET,
            "Model": name,
            "R2": r2_score(y_test, y_pred),
            "MAE": mean_absolute_error(y_test, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
            "MSE": mean_squared_error(y_test, y_pred)
        }

    results = []
    model_counter = 1
    total_models = 7

    def run_model(name, model, params):
        nonlocal model_counter

        log_progress(f"[{model_counter}/{total_models}] Starting {name}...")
        start_time = time.time()

        search = RandomizedSearchCV(
            model,
            params,
            n_iter=N_ITER_SEARCH,
            cv=CV_FOLDS,
            scoring="neg_root_mean_squared_error",
            random_state=SEED,
            n_jobs=-1,
            verbose=0
        )

        total_fits = N_ITER_SEARCH * CV_FOLDS

        with tqdm_joblib(tqdm(total=total_fits, desc=f"{TARGET} - {name} CV")):
            search.fit(X_train, y_train)

        log_progress(f"{name} best parameters: {search.best_params_}")
        results.append(evaluate(search.best_estimator_, name))

        log_progress(f"{name} finished in {time.time() - start_time:.2f} sec\n")
        model_counter += 1

    # -------------------------
    # MODELS
    # -------------------------

    run_model("RandomForest",
              RandomForestRegressor(random_state=SEED),
              {"n_estimators": [200, 300],
               "max_depth": [None, 10, 20, 30],
               "min_samples_split": [2, 5, 10],
               "min_samples_leaf": [1, 2, 4]})

    run_model("XGBoost",
              xgb.XGBRegressor(objective="reg:squarederror", random_state=SEED),
              {"n_estimators": [300, 400],
               "max_depth": [4, 6, 8],
               "learning_rate": [0.01, 0.05, 0.1],
               "subsample": [0.7, 0.9, 1.0],
               "colsample_bytree": [0.7, 0.9, 1.0]})

    run_model("LightGBM",
              lgb.LGBMRegressor(random_state=SEED, verbose=-1),
              {"n_estimators": [300, 600],
               "learning_rate": [0.05, 0.1],
               "num_leaves": [31, 63],
               "min_child_samples": [20, 50]})

    run_model("CatBoost",
              CatBoostRegressor(verbose=0, random_seed=SEED),
              {"depth": [4, 6, 8],
               "learning_rate": [0.01, 0.05, 0.1],
               "iterations": [500, 800, 1200]})

    run_model("SVR",
              SVR(),
              {"C": [1, 10, 100],
               "gamma": ["scale", 0.01, 0.001],
               "epsilon": [0.01, 0.1, 0.2]})

    run_model("KernelRidge",
              KernelRidge(),
              {"alpha": [0.01, 0.1, 1],
               "kernel": ["rbf", "poly"],
               "gamma": [0.01, 0.001]})

    run_model("MLP",
              MLPRegressor(random_state=SEED, max_iter=1000),
              {"hidden_layer_sizes": [(128,64), (256,128), (256,128,64)],
               "alpha": [1e-4, 1e-3, 1e-2],
               "learning_rate_init": [1e-3, 1e-4]})

    # -------------------------
    # Save Results
    # -------------------------
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"benchmark_results_{TARGET}.csv", index=False)

    log_progress(f"Finished TARGET: {TARGET}")
    print("\nResults for", TARGET)
    print(results_df)

    return results_df


# =====================================================
# RUN BOTH TARGETS
# =====================================================

all_results = []

for target in TARGETS:
    res = run_target(target)
    all_results.append(res)

final_df = pd.concat(all_results, ignore_index=True)
final_df.to_csv("benchmark_results_all_targets.csv", index=False)

log_progress("\nAll targets completed successfully.")